In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
# In Colab the package AND data/ must be present, so we clone the repo (Colab opens
# only the single .ipynb, not the repo). In CI/local the package is already installed
# from the checkout under test, so we skip the clone and exercise that code.
import os

try:
    import pocketlm  # already installed (local/CI) -> use the code under test
except ImportError:
    import subprocess
    import sys

    if not os.path.isdir("pocketlm-repo"):
        subprocess.check_call(
            ["git", "clone", "--depth", "1",
             "https://github.com/ni5h4nt/pocketlm", "pocketlm-repo"])
    os.chdir("pocketlm-repo")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
    import pocketlm  # noqa: F811

import torch

# nbmake runs a notebook from its own directory; anchor the cwd at the repo root
# (derived from the installed package) so CORPUS_PATH resolves in CI, locally, and
# in Colab (where the except-branch already chdir'd into the clone).
os.chdir(os.path.dirname(os.path.dirname(os.path.abspath(pocketlm.__file__))))

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = "data/corpus.txt"   # repo-root-relative; valid after the chdir above
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l3.1 Queries, keys and values

Attention is a soft, content-based lookup. Each token emits a **query** (what
am I looking for), a **key** (what do I offer), and a **value** (what I pass on).
Token i attends to token j in proportion to query_i . key_j.

In [ ]:
from torch import nn
from pocketlm import scaled_dot_product_attention

torch.manual_seed(0)
tokens, dim, hs = 3, 8, 8
x = torch.randn(tokens, dim)
w_q = nn.Linear(dim, hs, bias=False)
w_k = nn.Linear(dim, hs, bias=False)
w_v = nn.Linear(dim, hs, bias=False)
q, k, v = w_q(x), w_k(x), w_v(x)
out, weights = scaled_dot_product_attention(
    q.unsqueeze(0), k.unsqueeze(0), v.unsqueeze(0))
print("attention weights (each row sums to 1):")
print(weights[0].detach().round(decimals=2))

Each row is one token's distribution of attention over all tokens. The output
is the weighted sum of values, a blend of the information the token chose.

In [ ]:
assert torch.allclose(weights.sum(-1), torch.ones(1, tokens), atol=1e-6)